# Mass Spectrometry Analysis of Extracellular Vesicles Isolated from Stimulated Human Mast Cells Exploration with `mlcroissant`
This notebook provides an end-to-end template for loading, exploring, and processing the FAIR^2 dataset using the `mlcroissant` library.

### Dataset Source
The dataset source is provided via a [Croissant](https://mlcommons.org/croissant/) schema URL.

In [ ]:
# Ensure `mlcroissant` library is installed
!pip install --quiet mlcroissant

## 1. Data Loading
Load metadata and records from the dataset using `mlcroissant`.

In [ ]:
import mlcroissant as mlc
import pandas as pd

# Define the Croissant dataset URL
url = 'https://sen.science/doi/10.71728/senscience.vq4a-28xa/fair2.json'

# Load the dataset metadata
dataset = mlc.Dataset(url)
metadata = dataset.metadata
print(f"{metadata.name}: {metadata.description}")

## 2. Data Overview
Review available record sets, their fields, columns, and corresponding IDs (`@id`).

All referenceable components are shown by their Croissant `@id`. This allows you to select and reference datasets, record sets, and fields unambiguously.

In [ ]:
# List all RecordSets and their fields using their '@id'
from pprint import pprint
print('All RecordSets and their fields:')
record_set_ids = []

for record_set in metadata.record_sets:
    print(f"\nRecordSet name: {getattr(record_set, 'name', None)}")
    print(f"@id: {record_set.id}")
    record_set_ids.append(record_set.id)
    print('  Fields:')
    for field in record_set.fields:
        print(f"    - {getattr(field, 'name', None)} (@id: {field.id}, dataType: {getattr(field, 'data_type', None)})")

# Preview a record from each RecordSet
for record_set_id in record_set_ids:
    print(f"\nSample record from {record_set_id}:")
    try:
        for idx, rec in enumerate(dataset.records(record_set=record_set_id)):
            pprint(rec)
            if idx >= 0:
                break
    except Exception as e:
        print(f"  Could not read records from {record_set_id}: {e}")

## 3. Data Extraction
Load data from each available record set into a DataFrame for analysis. Collect fields and their IDs from the overview above.

In [ ]:
# Extract records from each record set by @id
all_dataframes = {}

for record_set_id in record_set_ids:
    try:
        records = list(dataset.records(record_set=record_set_id))
        df = pd.DataFrame(records)
        all_dataframes[record_set_id] = df
        print(f"RecordSet {record_set_id} loaded with columns: {df.columns.tolist()}")
    except Exception as e:
        print(f"Could not load {record_set_id}: {e}")

# Pick the primary table for further analysis (by default, use the first record set)
primary_record_set_id = record_set_ids[0]
print(f"\nUsing RecordSet: {primary_record_set_id} for analysis.")
df = all_dataframes[primary_record_set_id]
print(df.head())

## 4. Exploratory Data Analysis (EDA)
Apply typical data processing steps, including filtering records, normalizing numeric fields, and grouping (aggregation). All columns and fields are referenced by their Croissant `@id` where possible.

In [ ]:
import numpy as np

# Identify candidate numeric fields for processing
numeric_candidates = df.select_dtypes(include=[np.number]).columns.tolist()
print('Numeric fields:', numeric_candidates)
if not numeric_candidates:
    # If no numeric detected, attempt to cast likely numeric columns
    potential_numeric = [col for col in df.columns if 'count' in col.lower() or 'molecular_weight' in col.lower() or 'abundance' in col.lower() or 'coverage' in col.lower()]
    for col in potential_numeric:
        df[col] = pd.to_numeric(df[col], errors='coerce')
    numeric_candidates = df.select_dtypes(include=[np.number]).columns.tolist()
    print('Attempted to convert additional numeric fields. Now:', numeric_candidates)

# choose the first numeric field for demonstration
if numeric_candidates:
    numeric_field = numeric_candidates[0]
else:
    raise ValueError("No numeric field found for demonstration.")

threshold = df[numeric_field].mean() if df[numeric_field].notnull().any() else 0  # filter above mean
filtered_df = df[df[numeric_field] > threshold].copy()
print(f"Filtered records with {numeric_field} > {threshold:.2f}: {len(filtered_df)} records")
print(filtered_df[[numeric_field]].head())

# Normalize field
filtered_df[f"{numeric_field}_normalized"] = (filtered_df[numeric_field] - filtered_df[numeric_field].mean())/filtered_df[numeric_field].std()
print(f"Normalized {numeric_field} for filtered records:")
print(filtered_df[[numeric_field, f"{numeric_field}_normalized"]].head())

# Try grouping by a likely categorical field
categorical_candidates = [col for col in df.columns if df[col].dtype == 'object' and col != numeric_field]
group_field = None
for candidate in categorical_candidates:
    if df[candidate].nunique() > 1 and df[candidate].nunique() < 20:
        group_field = candidate
        break

if group_field:
    grouped_df = filtered_df.groupby(group_field)[numeric_field].mean().reset_index()
    print(f"\nGrouped data by {group_field} (mean of {numeric_field}):")
    print(grouped_df.head())
else:
    print('No suitable categorical group field found.')

## 5. Visualization
Visualize data distributions or relationships using simple plots. Make sure to reference fields and groupings by their Croissant `@id` or column name.

In [ ]:
import matplotlib.pyplot as plt
%matplotlib inline

# Histogram of the numeric field
plt.figure(figsize=(7,4))
filtered_df[numeric_field].hist(bins=30, alpha=0.7)
plt.title(f"Distribution of {numeric_field} (Filtered)")
plt.xlabel(numeric_field)
plt.ylabel("Frequency")
plt.show()

# If grouping exists, draw boxplots
if group_field:
    plt.figure(figsize=(9,4))
    filtered_df.boxplot(column=numeric_field, by=group_field, grid=False)
    plt.title(f"{numeric_field} by {group_field}")
    plt.suptitle('')
    plt.xlabel(group_field)
    plt.ylabel(numeric_field)
    plt.show()

## 6. Conclusion

This notebook demonstrated how to load and explore the FAIR² dataset using the Croissant schema and the `mlcroissant` Python library. By using the unique `@id` identifiers for record sets and fields, we ensured reproducible and robust access to data components.

- We loaded the dataset's metadata and discovered all available record sets and fields.
- Data was ingested for each record set and inspected in DataFrames.
- Simple exploratory data analysis (EDA) and normalization were applied to a numeric field, with results visualized where applicable.

You can further extend this notebook to perform protein-specific analyses, machine learning, or integrate with domain-specific bioinformatics tools.